# Session 9 Step 1: lambda bisection at the production point

Five-point bisection over lambda in {0.001, 0.003, 0.01, 0.03, 0.1} at (d=32, eta=0.01, OBS=cl_future, seed=0).
Anchors E4 (Session 8) and E5 (Session 7 R3) reused from disk; F1, F2, F3 are the new bisection points.
After lambda* is identified, F4 (seed=42) and F5 (seed=123) add the variance bound.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path('.').resolve()
S9 = REPO / 'outputs' / 'runs' / 'session9'
df = pd.read_csv(S9 / 'bisection_seed0.csv')
df.head()

## Test B delta vs lambda curve (seed=0)

The Session 8 grid (D53) sampled lambda in {0.01, 0.1, 1.0} at eta=0.01 and saw a monotone-decreasing pattern. The Session 9 bisection samples lambda in {0.001, 0.003, 0.01, 0.03, 0.1} to resolve whether lambda* is at the lower edge or interior.

In [ ]:
tb = df[(df['split'] == 'test_b') & (df['seed'] == 0)].sort_values('lambda').reset_index(drop=True)
fig, ax = plt.subplots(figsize=(7, 5))
ax.semilogx(tb['lambda'], tb['delta'], marker='o', linewidth=2)
for _, r in tb.iterrows():
    ax.annotate(r['code'], xy=(r['lambda'], r['delta']),
                xytext=(5, 5), textcoords='offset points')
ax.axhline(0.0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('lambda (SIGReg weight)')
ax.set_ylabel('delta_test_b = r2(z) - r2(c, t)')
ax.set_title('Lambda bisection at (d=32, eta=0.01), seed=0')
ax.grid(True, alpha=0.3)
fig.tight_layout()
tb

## Per-split metric table for all five bisection points

Each row is one (code, split) pair; the headline column is `delta`.

In [ ]:
summary = df[df['seed'] == 0].pivot_table(
    index=['code', 'lambda'],
    columns='split',
    values='delta',
).reset_index().sort_values('lambda')
summary

## PR_all and r2(z->c) across lambda

Looking at whether the optimum moves the latent regime.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.semilogx(tb['lambda'], tb['PR_all'], marker='o')
ax1.set_xlabel('lambda')
ax1.set_ylabel('PR_all (Test B)')
ax1.set_title('Participation ratio vs lambda')
ax1.grid(True, alpha=0.3)
ax2.semilogx(tb['lambda'], tb['r2_z_c'], marker='o')
ax2.set_xlabel('lambda')
ax2.set_ylabel('r2(z -> c) (Test B)')
ax2.set_title('Static-condition probe vs lambda')
ax2.grid(True, alpha=0.3)
fig.tight_layout()

## Seed-variance bound at lambda*

After Step 1 identifies lambda*, F4 (seed=42) and F5 (seed=123) are run at the same configuration. Pass criterion: best Test B delta within +/- 0.03 of the seed=0 result.

In [ ]:
best_path = S9 / 'best_lambda_star.json'
if best_path.exists():
    best = json.load(open(best_path))
    print(f"lambda* = {best['best_lambda']} ({best['best_code']}); seed=0 delta_test_b = {best['best_delta_test_b']:+.3f}")
else:
    print('best_lambda_star.json not yet written')

In [ ]:
variance_path = S9 / 'bisection_seed_variance.csv'
if variance_path.exists():
    vdf = pd.read_csv(variance_path)
    print('Seed-variance table at lambda*:')
    print(vdf[vdf['split'] == 'test_b'][['code', 'seed', 'lambda', 'delta']].round(3).to_string(index=False))
else:
    print('seed-variance not yet evaluated (F4/F5 still training)')

## Outcome summary (PRODUCTION_LOCKED / REFINED / PIVOT)

Per the Session 9 plan, the bisection outcome is one of three categories:
- PRODUCTION_LOCKED: lambda*=0.01 (same as Session 8). Production config unchanged.
- PRODUCTION_REFINED: lambda* in {0.001, 0.003, 0.03}. Production config updates; Section 5 gets a one-paragraph update.
- PRODUCTION_PIVOT: lambda* with delta > +0.20 OR seed-variance > +/- 0.05. Triggers deeper investigation.